# 👥 Multi-Agent Systems

**Coordinate multiple AI agents working together**

---

## 📋 Overview

**What you'll learn:**
- Multi-agent architectures
- Agent communication
- Hierarchical agents
- Collaborative problem solving
- Real-world examples

**Time estimate:** ⏱️ 60 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List
from dataclasses import dataclass

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Multiple Agents?

### Single Agent:
```
User: "Write and deploy a web application"

Agent: Must do everything:
  - Design architecture
  - Write frontend code
  - Write backend code
  - Set up database
  - Deploy to cloud

❌ Complex, error-prone, no specialization
```

### Multi-Agent:
```
User: "Write and deploy a web application"

Manager Agent: "Let me coordinate..."
  → Frontend Agent: Build UI
  → Backend Agent: Build API
  → Database Agent: Set up DB
  → DevOps Agent: Deploy

✅ Specialized, parallel, efficient
```

### Multi-Agent Patterns:

**1. Hierarchical** (Manager + Workers)
```
      Manager
     /   |   \
    A1   A2   A3
```

**2. Collaborative** (Peer-to-Peer)
```
A1 ←→ A2 ←→ A3
```

**3. Pipeline** (Sequential)
```
A1 → A2 → A3 → Result
```

## 🏗️ Basic Agent Class

In [ ]:
@dataclass
class AgentMessage:
    """Message between agents."""
    from_agent: str
    to_agent: str
    content: str
    message_type: str = "info"  # info, request, response

class Agent:
    """Base agent class."""
    
    def __init__(self, name: str, role: str, instructions: str):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.memory = []
    
    def process(self, message: str) -> str:
        """Process a message and return response."""
        
        # Build prompt
        messages = [
            {"role": "system", "content": f"""You are {self.name}, {self.role}.

{self.instructions}

Respond concisely and professionally."""},
            {"role": "user", "content": message}
        ]
        
        # Get response
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            temperature=0.7
        )
        
        result = response.choices[0].message.content
        
        # Store in memory
        self.memory.append({"input": message, "output": result})
        
        return result

# Create specialized agents
researcher = Agent(
    name="Researcher",
    role="Research Specialist",
    instructions="You research topics thoroughly and provide factual, well-sourced information."
)

writer = Agent(
    name="Writer",
    role="Content Writer",
    instructions="You write clear, engaging content based on research provided to you."
)

editor = Agent(
    name="Editor",
    role="Content Editor",
    instructions="You review and improve content for clarity, grammar, and flow."
)

print("👥 Created 3 specialized agents:")
print(f"  1. {researcher.name} - {researcher.role}")
print(f"  2. {writer.name} - {writer.role}")
print(f"  3. {editor.name} - {editor.role}")

## 🔄 Sequential Multi-Agent Pipeline

In [ ]:
class AgentPipeline:
    """Sequential agent pipeline."""
    
    def __init__(self, agents: List[Agent]):
        self.agents = agents
    
    def run(self, initial_input: str) -> str:
        """Run input through pipeline."""
        
        print(f"🔄 Pipeline with {len(self.agents)} agents\n")
        print("="*60)
        
        current_output = initial_input
        
        for i, agent in enumerate(self.agents, 1):
            print(f"\n📍 Stage {i}: {agent.name}")
            print(f"   Input: {current_output[:80]}...")
            
            current_output = agent.process(current_output)
            
            print(f"   Output: {current_output[:80]}...")
        
        return current_output

# Create pipeline
pipeline = AgentPipeline([
    researcher,  # Research topic
    writer,      # Write content
    editor       # Edit content
])

# Run pipeline
topic = "Explain what AI agents are in 100 words"
final_output = pipeline.run(topic)

print("\n" + "="*60)
print(f"\n✅ Final Output:\n{final_output}")

## 👔 Hierarchical Multi-Agent System

In [ ]:
class ManagerAgent(Agent):
    """Manager agent that coordinates workers."""
    
    def __init__(self, name: str, workers: List[Agent]):
        super().__init__(
            name=name,
            role="Project Manager",
            instructions="""You coordinate a team of specialists.
Break down tasks and delegate to the right team members.
Synthesize their results into a coherent final output."""
        )
        self.workers = {w.name: w for w in workers}
    
    def delegate(self, task: str) -> Dict[str, str]:
        """Break down task and delegate to workers."""
        
        print(f"\n👔 {self.name}: Planning task...\n")
        
        # Ask LLM to create delegation plan
        planning_prompt = f"""Break down this task for a team:

Task: {task}

Team members:
{', '.join(self.workers.keys())}

For each team member, specify what they should do.
Format as JSON:
{{
  "Researcher": "subtask for researcher",
  "Writer": "subtask for writer",
  ...
}}

JSON:"""
        
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": planning_prompt}],
            temperature=0
        )
        
        try:
            plan = json.loads(response.choices[0].message.content)
        except:
            plan = {name: task for name in self.workers.keys()}
        
        print("📋 Delegation Plan:")
        for worker, subtask in plan.items():
            print(f"  • {worker}: {subtask}")
        
        # Execute subtasks
        results = {}
        
        for worker_name, subtask in plan.items():
            if worker_name in self.workers:
                print(f"\n🤖 {worker_name} working...")
                result = self.workers[worker_name].process(subtask)
                results[worker_name] = result
                print(f"   ✅ Done: {result[:60]}...")
        
        return results
    
    def synthesize(self, results: Dict[str, str]) -> str:
        """Combine worker results."""
        
        print(f"\n👔 {self.name}: Synthesizing results...\n")
        
        synthesis_prompt = f"""Combine these team outputs into a cohesive result:

{json.dumps(results, indent=2)}

Create a well-structured, complete response."""
        
        return self.process(synthesis_prompt)
    
    def execute(self, task: str) -> str:
        """Execute task with team."""
        results = self.delegate(task)
        final_result = self.synthesize(results)
        return final_result

# Create hierarchical system
manager = ManagerAgent(
    name="Project Manager",
    workers=[researcher, writer, editor]
)

# Execute complex task
task = "Create a comprehensive guide about AI agents"
print(f"📝 Task: {task}")
print("="*60)

result = manager.execute(task)

print("\n" + "="*60)
print(f"\n✅ Final Result:\n{result}")

## 🤝 Collaborative Agents

In [ ]:
class CollaborativeSystem:
    """Agents collaborate peer-to-peer."""
    
    def __init__(self, agents: List[Agent]):
        self.agents = {a.name: a for a in agents}
        self.conversation = []
    
    def facilitate_discussion(self, topic: str, rounds: int = 3) -> str:
        """Facilitate multi-agent discussion."""
        
        print(f"🤝 Collaborative Discussion: {topic}\n")
        print("="*60)
        
        current_context = topic
        
        for round_num in range(rounds):
            print(f"\n📍 Round {round_num + 1}")
            
            round_contributions = []
            
            for agent_name, agent in self.agents.items():
                # Each agent contributes
                prompt = f"""Discuss: {topic}

Previous discussion:
{current_context}

Add your perspective (1-2 sentences):"""
                
                contribution = agent.process(prompt)
                round_contributions.append(f"{agent_name}: {contribution}")
                
                print(f"  {agent_name}: {contribution}")
            
            # Update context
            current_context += "\n\n" + "\n".join(round_contributions)
        
        # Synthesize final answer
        synthesis_prompt = f"""Based on this discussion, provide a final consensus:

{current_context}

Final answer:"""
        
        # Use first agent to synthesize
        final = list(self.agents.values())[0].process(synthesis_prompt)
        
        return final

# Create collaborative system
collab_system = CollaborativeSystem([researcher, writer, editor])

topic = "What are the most important considerations when building AI agents?"
result = collab_system.facilitate_discussion(topic, rounds=2)

print("\n" + "="*60)
print(f"\n✅ Consensus:\n{result}")

## ✅ Summary

### Multi-Agent Patterns:

**1. Sequential Pipeline**
```
Agent1 → Agent2 → Agent3 → Result

Use when:
- Clear workflow
- Each step depends on previous
- Example: Research → Write → Edit
```

**2. Hierarchical (Manager + Workers)**
```
       Manager
      /   |   \
     A1  A2   A3

Use when:
- Complex task needs decomposition
- Specialized agents
- Example: Project with multiple roles
```

**3. Collaborative (Peer-to-Peer)**
```
A1 ←→ A2 ←→ A3

Use when:
- Need diverse perspectives
- Consensus building
- Example: Brainstorming, decision making
```

### Benefits of Multi-Agent:

**✅ Specialization**
- Each agent has specific expertise
- Better at focused tasks

**✅ Modularity**
- Easy to add/remove agents
- Update individual agents

**✅ Parallelization**
- Multiple agents work simultaneously
- Faster execution

**✅ Robustness**
- If one agent fails, others continue
- Redundancy

### Implementation Tips:

**1. Clear Roles**
```python
agent = Agent(
    name="Researcher",
    role="Research Specialist",
    instructions="You research topics and provide factual information."
)
```

**2. Communication Protocol**
```python
@dataclass
class Message:
    from_agent: str
    to_agent: str
    content: str
    type: str  # "request", "response", "info"
```

**3. Orchestration**
```python
# Manager coordinates workflow
class Manager:
    def delegate(self, task):
        # Break down task
        # Assign to agents
        # Collect results
        # Synthesize
```

**4. Error Handling**
```python
# Handle agent failures
try:
    result = agent.process(task)
except AgentError:
    result = fallback_agent.process(task)
```

### When to Use Multi-Agent:

✅ **Use multi-agent for:**
- Complex, multi-step tasks
- Need for specialization
- Parallel processing
- Diverse perspectives needed

❌ **Single agent is better for:**
- Simple tasks
- Low latency requirements
- Limited resources
- Straightforward workflows

### Real-World Examples:

**Software Development Team:**
```python
manager = ManagerAgent(workers=[
    frontend_agent,
    backend_agent,
    database_agent,
    testing_agent,
    devops_agent
])
```

**Content Creation:**
```python
pipeline = Pipeline([
    research_agent,
    outline_agent,
    writing_agent,
    editing_agent,
    fact_check_agent
])
```

**Customer Support:**
```python
router_agent → classifier
            ↓
        [billing_agent, technical_agent, sales_agent]
```

### Congratulations! 🎉

You've completed the Agents & Tools module!

**Next module:** `08_production_apis/` - Building production-ready APIs